In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.fake_provider import FakeWashingtonV2
from matplotlib import pyplot as plt

def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
        return qc
    else:
        raise ValueError("n must be an integer >= 2")

n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(op) for op in operator_strings]

backend = FakeWashingtonV2()
estimator = Estimator(backend)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

mapped_operators = [
    op.apply_layout(isa_circuit.layout) for op in operators
]

job = estimator.run([(isa_circuit, mapped_operators)])
result = job.result()[0]

values = result.data.evs
values = [v / values[0] for v in values]

data = list(range(1, len(values) + 1))

plt.plot(data, values, marker="o", label="100-qubit GHZ state (simulated)")
plt.xlabel("Distance between qubits i")
plt.ylabel("<Zi Z0> / <Z1 Z0>")
plt.legend()
plt.show()
